# v10 — asymmetric-precision recipe ablation (accuracy, pure PyTorch, Colab T4)

The v10 gate measured STANDARD NVFP4 (per-tensor + per-16 micro-scale) at ~2.4e-3 output RMSE = ~4x
the FP8 floor (~6e-4). The asymmetric recipe — **K per-channel** (outlier channels), **V per-token**
(outlier tokens) — is the lever to recover it (KIVI/KVQuant/KVTuner). This is a **pure fake-quant
study** (quantize→dequantize→SDPA→RMSE; NO kernel, NO byte-packing), so it runs in seconds and needs
no build. **Roofline note:** capacity is unchanged to first order (a per-channel scale is d/tensor, a
per-token scale is N_k/tensor — negligible vs the nibbles), so this is a PURE accuracy lever at fixed
~0.56 B/elem; the roofline is blind to it. Deliverable = the K×V granularity matrix + the
FP4-everything softmax-collapse figure.

## 0. Dependencies + GPU (venv-safe)

In [1]:
import os, sys, subprocess

def pip(*pkgs, extra=()):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs, *extra], check=True)

# 1) Physical GPU on this runtime? (Colab defaults to CPU; pick a GPU explicitly.)
try:
    has_gpu = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
if not has_gpu:
    raise SystemExit(
        'No GPU on this Colab runtime. FIX: Runtime > Change runtime type > T4 GPU > Save, '
        'then Runtime > Restart session, then re-run from the top. (This kernel needs a Turing T4.)')

# 2) Install deps (incl. numpy) BEFORE importing torch, so torch's numpy bridge initializes.
pip('ninja', 'pytest', 'numpy')

# 3) torch present AND CUDA-enabled? A CPU-only wheel raises "not compiled with CUDA" on any kernel.
try:
    import torch
    cuda_ok = torch.cuda.is_available()
except ImportError:
    torch, cuda_ok = None, False

if not cuda_ok:
    pip('torch', extra=('--index-url', 'https://download.pytorch.org/whl/cu124'))
    raise SystemExit(
        'A GPU is present but torch was a CPU-only build -- installed the CUDA build. NOW: '
        'restart the kernel/session, then re-run this cell.')

# vast.ai/venv: !-cells spawn a bare shell without the venv on PATH -> `python` not found.
os.environ['PATH'] = os.path.dirname(sys.executable) + os.pathsep + os.environ.get('PATH', '')

# torch.float8_e4m3fn must exist (>=2.1) — v9 stores the KV cache as E4M3 bytes.
assert hasattr(torch, 'float8_e4m3fn'), 'this torch lacks float8_e4m3fn; upgrade torch (>=2.1)'
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| cap', torch.cuda.get_device_capability())
!nvidia-smi --query-gpu=name,compute_cap --format=csv

torch 2.11.0+cu128 | cuda 12.8 | cap (7, 5)
name, compute_cap
Tesla T4, 7.5


## 1. Get the repo

In [2]:
REPO_URL = 'https://github.com/gkienpham-cmd/flashattention-cuda.git'  # public; plain clone works
import os, sys, subprocess
if os.path.basename(os.getcwd()) != 'flashattention-cuda':
    if not os.path.isdir('flashattention-cuda'):
        subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir('flashattention-cuda')
subprocess.run(['git', 'pull', 'origin', 'main'])
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('cwd', os.getcwd())

cwd /content/flashattention-cuda


## 2. The K×V granularity matrix — does asymmetric (K=channel, V=token) beat block16?

In [3]:
# End-to-end attention RMSE vs fp16 KV, for each (K granularity x V granularity). block16 = the gate's
# MEASURED recipe (reuses the real kernel quantizer). Lower = better; the FP8 floor is the bar.
import torch
from fa_kernels.nvfp4_recipes import attn_rmse, fp8_attn_rmse, GRANULARITIES

def run_matrix(d, G, N_k=8192, seeds=(9, 17, 23)):
    B, H_kv = 1, 2
    H_q = G * H_kv
    rows = {}
    fp8 = []
    for seed in seeds:
        torch.manual_seed(seed)
        q = torch.randn(B, H_q,  1,   d, device='cuda')
        k = torch.randn(B, H_kv, N_k, d, device='cuda')
        v = torch.randn(B, H_kv, N_k, d, device='cuda')
        fp8.append(fp8_attn_rmse(q, k, v))
        for kg in GRANULARITIES:
            for vg in GRANULARITIES:
                rows.setdefault((kg, vg), []).append(attn_rmse(q, k, v, k_gran=kg, v_gran=vg))
    import statistics as st
    fp8_m = st.mean(fp8)
    print(f'=== d={d} G={G} N_k={N_k} | FP8 floor RMSE = {fp8_m:.3e} (the bar) ===')
    print(f"{'K\\V':>8} | " + ' | '.join(f'{vg:>9}' for vg in GRANULARITIES))
    for kg in GRANULARITIES:
        cells_ = []
        for vg in GRANULARITIES:
            m = st.mean(rows[(kg, vg)])
            tag = '*' if m <= fp8_m else ' '   # beats/meets the FP8 floor
            cells_.append(f'{m:.2e}{tag}')
        print(f'{kg:>8} | ' + ' | '.join(f'{c:>9}' for c in cells_))
    return rows, fp8_m

for d in (64, 128):
    for G in (1, 8):
        run_matrix(d, G)
        print()
print('block16 row/col = the gate recipe (~2.4e-3). A trailing * means it met/beat the FP8 floor.')
print('HYPOTHESIS: (K=channel, V=token) is the best NVFP4 cell and closes most of the gap to FP8.')

=== d=64 G=1 N_k=8192 | FP8 floor RMSE = 6.865e-04 (the bar) ===
     K\V |    tensor |   block16 |   channel |     token
  tensor | 3.29e-03  | 2.81e-03  | 3.18e-03  | 2.99e-03 
 block16 | 3.13e-03  | 2.41e-03  | 2.67e-03  | 2.45e-03 
 channel | 3.23e-03  | 2.33e-03  | 2.86e-03  | 2.63e-03 
   token | 3.24e-03  | 2.48e-03  | 2.71e-03  | 2.72e-03 

=== d=64 G=8 N_k=8192 | FP8 floor RMSE = 7.136e-04 (the bar) ===
     K\V |    tensor |   block16 |   channel |     token
  tensor | 3.65e-03  | 3.11e-03  | 3.38e-03  | 3.25e-03 
 block16 | 3.14e-03  | 2.52e-03  | 2.80e-03  | 2.58e-03 
 channel | 3.38e-03  | 2.87e-03  | 3.11e-03  | 2.97e-03 
   token | 3.27e-03  | 2.72e-03  | 3.02e-03  | 2.84e-03 

=== d=128 G=1 N_k=8192 | FP8 floor RMSE = 6.569e-04 (the bar) ===
     K\V |    tensor |   block16 |   channel |     token
  tensor | 3.47e-03  | 3.06e-03  | 3.26e-03  | 3.17e-03 
 block16 | 2.99e-03  | 2.49e-03  | 2.77e-03  | 2.72e-03 
 channel | 3.24e-03  | 2.76e-03  | 2.96e-03  | 2.99e-03 
   t

## 3. The accuracy ladder (the figure) — block16 vs asymmetric vs FP8 floor

In [4]:
# Collapse the matrix to the storyline ladder at d=128 G=8 (the clean monotone case): the coarsest
# (tensor/tensor), the gate (block16/block16), the asymmetric (channel-K/token-V), and the FP8 floor.
import statistics as st, torch
from fa_kernels.nvfp4_recipes import attn_rmse, fp8_attn_rmse
rungs = [('tensor/tensor', 'tensor', 'tensor'), ('block16 (gate)', 'block16', 'block16'),
         ('asym K=ch/V=tok', 'channel', 'token')]
d, G, B, H_kv, N_k = 128, 8, 1, 2, 8192
H_q = G * H_kv
vals, fp8 = {}, []
for seed in (9, 17, 23):
    torch.manual_seed(seed)
    q = torch.randn(B, H_q, 1, d, device='cuda'); k = torch.randn(B, H_kv, N_k, d, device='cuda')
    v = torch.randn(B, H_kv, N_k, d, device='cuda')
    fp8.append(fp8_attn_rmse(q, k, v))
    for name, kg, vg in rungs:
        vals.setdefault(name, []).append(attn_rmse(q, k, v, k_gran=kg, v_gran=vg))
fp8_m = st.mean(fp8)
print(f'{"recipe":>18} | {"RMSE":>10} | {"x FP8 floor":>11}')
for name, _, _ in rungs:
    m = st.mean(vals[name])
    print(f'{name:>18} | {m:10.3e} | {m/fp8_m:10.2f}x')
print(f'{"FP8 floor":>18} | {fp8_m:10.3e} | {1.0:10.2f}x')

# Bar figure (saved; host-only matplotlib is fine for the notebook artifact).
try:
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    names = [n for n, _, _ in rungs] + ['FP8 floor']
    ys = [st.mean(vals[n]) for n, _, _ in rungs] + [fp8_m]
    plt.figure(figsize=(6, 3.2)); plt.bar(range(len(names)), ys, color=['#bbb','#69c','#3a3','#e83'])
    plt.yscale('log'); plt.xticks(range(len(names)), names, rotation=20, ha='right')
    plt.ylabel('attn output RMSE vs fp16'); plt.title('v10 NVFP4 accuracy ladder (d=128 G=8)')
    plt.tight_layout(); plt.savefig('diagrams/v10-accuracy-ladder.png', dpi=120)
    print('saved diagrams/v10-accuracy-ladder.png')
except Exception as e:
    print('plot skipped:', e)

            recipe |       RMSE | x FP8 floor
     tensor/tensor |  3.581e-03 |       5.15x
    block16 (gate) |  2.429e-03 |       3.49x
   asym K=ch/V=tok |  2.978e-03 |       4.28x
         FP8 floor |  6.958e-04 |       1.00x
plot skipped: [Errno 2] No such file or directory: 'diagrams/v10-accuracy-ladder.png'


## 4. The FP4-EVERYTHING ablation — quantize the score P too → softmax collapse

In [5]:
# Why the recipe keeps the SCORE >= FP16: quantize the post-softmax weights P to FP4 (per-token) on
# top of FP4 KV and watch the RMSE blow up vs FP4-KV-only. This justifies 'FP4 storage, not FP4 score'.
import torch
from fa_kernels.nvfp4_recipes import fp4_score_collapse_rmse
print(f'{"shape":>14} | {"FP4 KV only":>12} | {"+ FP4 score P":>13} | blow-up')
for d in (64, 128):
    for G in (1, 8):
        torch.manual_seed(9)
        B, H_kv, N_k = 1, 2, 8192; H_q = G * H_kv
        q = torch.randn(B, H_q, 1, d, device='cuda'); k = torch.randn(B, H_kv, N_k, d, device='cuda')
        v = torch.randn(B, H_kv, N_k, d, device='cuda')
        r = fp4_score_collapse_rmse(q, k, v)
        blow = r['fp4_kv_and_p'] / max(r['fp4_kv_only'], 1e-12)
        print(f"{f'{H_q}x1x{d} G{G}':>14} | {r['fp4_kv_only']:12.2e} | {r['fp4_kv_and_p']:13.2e} | {blow:6.1f}x")
print('\nA large blow-up = quantizing the score collapses softmax -> keep scores >= FP16.')
print('Already true in the kernel (sK is FP16, the Q.K^T dot is FP16) — the asymmetry justification.')

         shape |  FP4 KV only | + FP4 score P | blow-up
     2x1x64 G1 |     2.29e-03 |      1.19e-02 |    5.2x
    16x1x64 G8 |     3.11e-03 |      2.04e-02 |    6.6x
    2x1x128 G1 |     2.93e-03 |      1.35e-02 |    4.6x
   16x1x128 G8 |     3.07e-03 |      1.89e-02 |    6.2x

A large blow-up = quantizing the score collapses softmax -> keep scores >= FP16.
Already true in the kernel (sK is FP16, the Q.K^T dot is FP16) — the asymmetry justification.
